# PH2_NB6f — Exhaustive Feature-Subset Search over 13 Candidates

The first ablation searched 27 subsets of five features. This one searches **7,814 subsets of
thirteen**: the two features that proved to carry the load (`burstiness`, `ttr`) plus the eleven new
features from PH2_NB6e. The three near-inert originals — `targeted_ppl`, `entity_density`,
`discourse_coherence` — are excluded from the search space, as agreed, but return as mandatory
reference rows at the end so their absence is a measured decision rather than an assumption.

**Inputs:** `aigt-dataset`, `aigt-vstat16` (vstat16_scaled.parquet from NB6e), `aigt-camelbert-cls`.
**Outputs:** `ph2_nb6f_screen.parquet` (all configs, validation scores), `ph2_nb6f_final.parquet`
(finalists + references, test scores), `ph2_nb6f_marginals.parquet`.

## The selection-bias problem, and the two-stage protocol that solves it

Picking the best of 7,814 subsets **on the same folds we then report** would be textbook overfitting
of the evaluation: the GPT LOGO fold holds only 68 AI articles, so the winner of a 7,814-way contest
on that fold is mostly luck. With 27 candidates this was a caution; with 7,814 it would invalidate
the number.

So the test split is not touched during the search:

- **Stage 1 — screening (all 7,814).** LOGO is run with the held-out generator evaluated on the
  **validation** partition. Fast solver settings, checkpointed every 200 configs.
- **Stage 2 — reporting (finalists only).** The top configurations by validation, plus the mandatory
  references, are re-fit with full convergence and scored on the **test** split, standard + 6-fold
  LOGO. These are the numbers that go in the thesis.

Aggregated marginal contributions are computed over all 7,814 validation results — an average across
thousands of contexts, which is robust to selection effects in a way a single winner is not.

## Cost

7,814 configs x 6 validation fits at screening settings is roughly 4-6 hours on CPU, inside the
12-hour Kaggle limit, and the checkpoint makes it resumable. `MIN_ACTIVE` controls the search size if
a shorter run is wanted (raising it to 6 cuts the space to about 5,200).

## Setup

In [1]:
import pandas as pd, numpy as np, os, glob, time, json, itertools
from math import comb

SEED = 42
np.random.seed(SEED)
OUT_DIR = '/kaggle/working'

MIN_ACTIVE   = 4        # keep at least this many features (drop up to 13 - MIN_ACTIVE = 9)
TOP_K        = 25       # finalists per validation criterion carried into stage 2
SCREEN_ITER  = 1000     # screening solver budget
SCREEN_TOL   = 1e-3
FINAL_ITER   = 5000     # full convergence for reported numbers
FINAL_TOL    = 1e-4
CKPT_EVERY   = 200
RESUME       = None     # e.g. '/kaggle/input/aigt-nb6f-partial/ph2_nb6f_screen.parquet'

def find_file(preferred, pattern, *keywords):
    if os.path.exists(preferred): return preferred
    for kw in keywords:
        hits = [p for p in glob.glob(f'/kaggle/input/**/{pattern}', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    raise FileNotFoundError(preferred)

DATA  = find_file('/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet', '*.parquet', 'dataset')
V16   = find_file('/kaggle/input/notebooks/bahaaqassem/ph2-nb6e-extract-11-features/vstat16_scaled.parquet', '*.parquet', 'vstat16')
EMB   = find_file('/kaggle/input/notebooks/bahaaqassem/nb5c-electra-camelbert/nb5c_camelbert_msa_cls_frozen.npy', '*.npy',
                  'camelbert_msa_cls', 'camelbert')

df  = pd.read_parquet(DATA)
v16 = pd.read_parquet(V16)
emb = np.load(EMB).astype(np.float32)

OLD5 = ['targeted_ppl', 'burstiness', 'ttr', 'entity_density', 'discourse_coherence']
NEW11 = ['quote_ratio', 'quote_variety', 'coord_sub_ratio', 'function_word_ratio',
         'pos_entropy', 'passive_ratio', 'clause_depth', 'compressibility',
         'char_ngram_repetition', 'zipf_deviation', 'sent_opener_diversity']
ALL16 = OLD5 + NEW11
INERT = ['targeted_ppl', 'entity_density', 'discourse_coherence']     # excluded: near-inert in the 5-feature ablation
# Excluded as a near-duplicate: char_ngram_repetition correlates -0.96 with compressibility and is
# the weakest of all sixteen (class gap 0.026, AUC 0.526). Keeping both would make the search's
# choice between them a coin flip on noise, so the weaker twin is dropped before the search.
REDUNDANT = ['char_ngram_repetition']
POOL  = [f for f in ALL16 if f not in INERT + REDUNDANT]              # 12 candidates

assert len(emb) == len(df)
v16 = v16.set_index('article_id').loc[df['article_id']].reset_index()
assert (v16['article_id'].to_numpy() == df['article_id'].to_numpy()).all(), 'vstat16 misalignment'
assert (v16['label'].to_numpy() == df['label'].to_numpy()).all(), 'label mismatch'
assert v16[ALL16].isna().sum().sum() == 0

Xall16 = v16[ALL16].to_numpy(dtype=np.float32)
COL = {f: i for i, f in enumerate(ALL16)}
y = df['label'].to_numpy(); splits = df['split'].to_numpy(); gens = df['generator'].to_numpy()
gen_list = sorted(df.loc[df['label']==1, 'generator'].unique().tolist())

n_space = sum(comb(len(POOL), k) for k in range(MIN_ACTIVE, len(POOL)+1))
print(f'ALIGNMENT OK | emb {emb.shape} | 16 features | pool of {len(POOL)}: {POOL}')
print(f'excluded (inert, kept as references): {INERT}')
print(f'excluded (near-duplicate of compressibility): {REDUNDANT}')
print(f'search space with MIN_ACTIVE={MIN_ACTIVE}: {n_space} configurations')

ALIGNMENT OK | emb (7101, 768) | 16 features | pool of 12: ['burstiness', 'ttr', 'quote_ratio', 'quote_variety', 'coord_sub_ratio', 'function_word_ratio', 'pos_entropy', 'passive_ratio', 'clause_depth', 'compressibility', 'zipf_deviation', 'sent_opener_diversity']
excluded (inert, kept as references): ['targeted_ppl', 'entity_density', 'discourse_coherence']
excluded (near-duplicate of compressibility): ['char_ngram_repetition']
search space with MIN_ACTIVE=4: 3797 configurations


## Pool audit — collinearity inside the search space

Before spending the search on it, the pool is checked for internal duplication. Two features
correlating above 0.9 carry the same information, so whichever one a winning subset happens to
contain is arbitrary; the printout below makes any such pair visible so the results are read with
that in mind. `char_ngram_repetition` was already removed for exactly this reason.

In [2]:
C = v16[POOL].corr()
pairs = [(a, b, C.loc[a, b]) for i, a in enumerate(POOL) for b in POOL[i+1:]
         if abs(C.loc[a, b]) > 0.45]
print('collinear pairs inside the pool (|r| > 0.45):')
for a, b, r in sorted(pairs, key=lambda t: -abs(t[2])):
    tag = 'DUPLICATE' if abs(r) > 0.9 else ('high' if abs(r) > 0.7 else 'moderate')
    print(f'  {a:<24} x {b:<24} r={r:+.2f}  [{tag}]')
if not pairs:
    print('  none')

gaps = (v16[v16.label==1][POOL].mean() - v16[v16.label==0][POOL].mean()).abs().sort_values(ascending=False)
print('\nclass separation of the pool (|scaled mean gap|):')
print(gaps.round(3).to_string())

collinear pairs inside the pool (|r| > 0.45):
  compressibility          x zipf_deviation           r=+0.88  [high]
  passive_ratio            x clause_depth             r=+0.63  [moderate]
  ttr                      x zipf_deviation           r=+0.59  [moderate]
  function_word_ratio      x pos_entropy              r=+0.57  [moderate]
  compressibility          x sent_opener_diversity    r=+0.48  [moderate]
  zipf_deviation           x sent_opener_diversity    r=+0.47  [moderate]

class separation of the pool (|scaled mean gap|):
burstiness               0.973
function_word_ratio      0.700
ttr                      0.666
quote_ratio              0.583
pos_entropy              0.513
quote_variety            0.467
sent_opener_diversity    0.415
coord_sub_ratio          0.272
zipf_deviation           0.226
clause_depth             0.190
passive_ratio            0.179
compressibility          0.168


## Fitting helpers

`fit_eval` trains the converged linear head — the attribution-grade arm from the earlier ablation,
deterministic so subset-to-subset differences are real. Screening uses a looser solver budget;
finalists are re-fit at full convergence.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix

def build_X(feats):
    if not feats: return emb
    idx = [COL[f] for f in feats]
    return np.concatenate([emb, Xall16[:, idx]], axis=1)

def fit_eval(X, tr_mask, ev_mask, max_iter, tol):
    m = LogisticRegression(max_iter=max_iter, tol=tol,
                           class_weight='balanced', random_state=SEED)
    m.fit(X[tr_mask], y[tr_mask])
    proba = m.predict_proba(X[ev_mask])[:, 1]
    pred = (proba >= 0.5).astype(int)
    yy = y[ev_mask]
    return f1_score(yy, pred, average='macro'), roc_auc_score(yy, proba), pred

def logo_masks(g, eval_split):
    tr = (splits=='train') & ((y==0) | (gens!=g))
    ev = (splits==eval_split) & ((y==0) | (gens==g))
    return tr, ev

def screen_config(feats):
    # LOGO scored on the VALIDATION partition; the test split is untouched here
    X = build_X(feats)
    fs = []
    for g in gen_list:
        tr, ev = logo_masks(g, 'val')
        f1, _, _ = fit_eval(X, tr, ev, SCREEN_ITER, SCREEN_TOL)
        fs.append(f1)
    fs = np.array(fs)
    return {'val_logo_mean': fs.mean(), 'val_logo_worst': fs.min(),
            'val_logo_std': fs.std(),
            **{f'valfold_{g}': fs[i] for i, g in enumerate(gen_list)}}

def report_config(feats):
    # full convergence, standard TEST split + 6-fold TEST LOGO
    X = build_X(feats)
    tr, ev = (splits=='train'), (splits=='test')
    std_f1, std_auc, _ = fit_eval(X, tr, ev, FINAL_ITER, FINAL_TOL)
    fs, ai_rec = [], {}
    for g in gen_list:
        trm, evm = logo_masks(g, 'test')
        f1, _, pred = fit_eval(X, trm, evm, FINAL_ITER, FINAL_TOL)
        fs.append(f1)
        cm = confusion_matrix(y[evm], pred, labels=[0,1])
        ai_rec[g] = cm[1,1] / max(cm[1].sum(), 1)
    fs = np.array(fs)
    return {'test_standard_f1': std_f1, 'test_standard_auc': std_auc,
            'test_logo_mean': fs.mean(), 'test_logo_worst': fs.min(),
            'test_worst_gen': gen_list[int(fs.argmin())],
            **{f'testfold_{g}': fs[i] for i, g in enumerate(gen_list)},
            **{f'airecall_{g}': ai_rec[g] for g in gen_list}}

print('fitters ready | screening on val, reporting on test')

fitters ready | screening on val, reporting on test


## Stage 1 — screen all configurations on validation

Enumerated largest-first so the informative full-pool rows appear early in the log. The partial
table is written every 200 configurations; set `RESUME` to it and re-run if the session ends.

In [4]:
configs = []
for k in range(len(POOL), MIN_ACTIVE - 1, -1):
    for combo in itertools.combinations(POOL, k):
        configs.append(tuple(combo))
print('configurations to screen:', len(configs))

done = {}
if RESUME:
    prev = pd.read_parquet(find_file(RESUME, '*.parquet', 'nb6f_screen'))
    done = {r['key']: r.to_dict() for _, r in prev.iterrows()}
    print('resuming with', len(done), 'already screened')

rows = list(done.values())
t0 = time.time()
for i, feats in enumerate(configs):
    key = '|'.join(feats)
    if key in done: continue
    r = screen_config(list(feats))
    rows.append({'key': key, 'n_active': len(feats), **r})
    n = len(rows)
    if n % CKPT_EVERY == 0 or i == len(configs) - 1:
        pd.DataFrame(rows).to_parquet(f'{OUT_DIR}/ph2_nb6f_screen.parquet', index=False)
        el = time.time() - t0
        rate = el / max(n - len(done), 1)
        eta = rate * (len(configs) - n) / 60
        print(f'[{n:5d}/{len(configs)}] {rate:.2f}s/config | ETA {eta:.0f}m', flush=True)

screen = pd.DataFrame(rows)
screen.to_parquet(f'{OUT_DIR}/ph2_nb6f_screen.parquet', index=False)
print(f'\nscreening complete: {screen.shape} in {(time.time()-t0)/60:.0f} min')

configurations to screen: 3797
[  200/3797] 1.53s/config | ETA 91m
[  400/3797] 1.52s/config | ETA 86m
[  600/3797] 1.53s/config | ETA 81m
[  800/3797] 1.52s/config | ETA 76m
[ 1000/3797] 1.51s/config | ETA 70m
[ 1200/3797] 1.52s/config | ETA 66m
[ 1400/3797] 1.51s/config | ETA 60m
[ 1600/3797] 1.51s/config | ETA 55m
[ 1800/3797] 1.50s/config | ETA 50m
[ 2000/3797] 1.51s/config | ETA 45m
[ 2200/3797] 1.50s/config | ETA 40m
[ 2400/3797] 1.49s/config | ETA 35m
[ 2600/3797] 1.49s/config | ETA 30m
[ 2800/3797] 1.48s/config | ETA 25m
[ 3000/3797] 1.47s/config | ETA 20m
[ 3200/3797] 1.47s/config | ETA 15m
[ 3400/3797] 1.47s/config | ETA 10m
[ 3600/3797] 1.47s/config | ETA 5m
[ 3797/3797] 1.47s/config | ETA 0m

screening complete: (3797, 11) in 93 min


## Aggregated marginal contribution (over all 7,814 validation results)

For each candidate: mean validation score across every subset containing it, minus the mean across
every subset lacking it. Averaged over thousands of contexts, this is the robust statement about a
feature's worth — unlike any single winning subset.

In [5]:
marg = []
key_sets = screen['key'].apply(lambda s: set(s.split('|')))
for f in POOL:
    has = screen[key_sets.apply(lambda S: f in S)]
    non = screen[key_sets.apply(lambda S: f not in S)]
    if len(non) == 0:
        continue
    marg.append({'feature': f, 'n_with': len(has), 'n_without': len(non),
                 'm_val_worst': 100*(has['val_logo_worst'].mean() - non['val_logo_worst'].mean()),
                 'm_val_mean':  100*(has['val_logo_mean'].mean()  - non['val_logo_mean'].mean())})
marg = pd.DataFrame(marg).sort_values('m_val_worst', ascending=False)
print('aggregated marginal contribution on validation (pp; + = helps on average):')
print(marg.round(3).to_string(index=False))
marg.to_parquet(f'{OUT_DIR}/ph2_nb6f_marginals.parquet', index=False)

aggregated marginal contribution on validation (pp; + = helps on average):
              feature  n_with  n_without  m_val_worst  m_val_mean
           burstiness    1981       1816        0.972       0.097
                  ttr    1981       1816        0.823       0.628
       zipf_deviation    1981       1816        0.213       0.036
        passive_ratio    1981       1816        0.182      -0.018
  function_word_ratio    1981       1816        0.094       0.341
      compressibility    1981       1816        0.092       0.006
          quote_ratio    1981       1816        0.064       0.010
          pos_entropy    1981       1816       -0.009       0.026
        quote_variety    1981       1816       -0.209      -0.033
         clause_depth    1981       1816       -0.236      -0.103
sent_opener_diversity    1981       1816       -0.964       0.016
      coord_sub_ratio    1981       1816       -1.851      -0.302


## Stage 2 — finalists and mandatory references, scored on the test split

Finalists are the top `TOP_K` by validation worst fold, unioned with the top `TOP_K` by validation
mean (two criteria, so a subset that is merely lucky on one fold does not automatically win). The
mandatory references are evaluated regardless:

- the current official five (the model the thesis currently claims),
- the full pool of thirteen,
- all sixteen,
- and the best finalist **plus `targeted_ppl`**, so the cost of leaving perplexity out is measured
  rather than assumed — it is named in the proposal's title and keywords.

In [6]:
top_worst = screen.sort_values(['val_logo_worst','val_logo_mean'], ascending=False).head(TOP_K)
top_mean  = screen.sort_values(['val_logo_mean','val_logo_worst'], ascending=False).head(TOP_K)
finalist_keys = list(dict.fromkeys(top_worst['key'].tolist() + top_mean['key'].tolist()))
print(f'finalists carried to test: {len(finalist_keys)}')

best_key = top_worst.iloc[0]['key']
best_feats = best_key.split('|')

references = {
    'REF official 5':          OLD5,
    'REF pool 13':             POOL,
    'REF all 16':              ALL16,
    'REF best + targeted_ppl': sorted(set(best_feats) | {'targeted_ppl'}),
    'REF neural only':         [],
}

final_rows = []
for k in finalist_keys:
    feats = k.split('|')
    final_rows.append({'label': 'search: ' + ', '.join(feats), 'n_active': len(feats),
                       'key': k, 'is_reference': False, **report_config(feats)})
for name, feats in references.items():
    final_rows.append({'label': name, 'n_active': len(feats),
                       'key': '|'.join(feats), 'is_reference': True, **report_config(feats)})

final = pd.DataFrame(final_rows).sort_values(['test_logo_worst','test_standard_f1'], ascending=False)
final.to_parquet(f'{OUT_DIR}/ph2_nb6f_final.parquet', index=False)

show = final[['label','n_active','test_standard_f1','test_logo_mean','test_logo_worst','test_worst_gen']].copy()
for c in ['test_standard_f1','test_logo_mean','test_logo_worst']:
    show[c] = (100*show[c]).round(1)
print('\nTEST results (finalists + references), ranked by LOGO worst fold:\n')
print(show.head(20).to_string(index=False))
print('\nreferences:')
print(show[final['is_reference'].values].to_string(index=False))

finalists carried to test: 29

TEST results (finalists + references), ranked by LOGO worst fold:

                                                                                                                  label  n_active  test_standard_f1  test_logo_mean  test_logo_worst test_worst_gen
                              search: burstiness, ttr, quote_ratio, function_word_ratio, passive_ratio, compressibility         6              99.6            98.2             95.2            gpt
                                           search: burstiness, ttr, function_word_ratio, passive_ratio, compressibility         5              99.7            98.2             94.7            gpt
                  search: burstiness, ttr, quote_ratio, function_word_ratio, pos_entropy, clause_depth, compressibility         7              99.7            98.2             94.7            gpt
               search: burstiness, ttr, quote_ratio, quote_variety, function_word_ratio, passive_ratio, compressibilit

## Verdict

The comparison that matters: the best searched subset against the current official five, both on the
untouched test split. A subset only earns promotion if it beats the official model on the
pre-registered criterion (LOGO worst fold) **and** holds standard-split parity — and even then, the
validation-to-test gap should be small, because a large gap means the validation screening chased
noise despite the protocol.

In [7]:
off = final[final['label']=='REF official 5'].iloc[0]
best = final[~final['is_reference']].iloc[0]
best_val = screen[screen['key']==best['key']].iloc[0]

print('official five     : std {:.1f} | logo mean {:.1f} | worst {:.1f} ({})'.format(
    100*off['test_standard_f1'], 100*off['test_logo_mean'],
    100*off['test_logo_worst'], off['test_worst_gen']))
print('best searched     : std {:.1f} | logo mean {:.1f} | worst {:.1f} ({})'.format(
    100*best['test_standard_f1'], 100*best['test_logo_mean'],
    100*best['test_logo_worst'], best['test_worst_gen']))
print('  features        :', best['label'].replace('search: ', ''))
print('  n_active        :', best['n_active'])

gap = 100*(best_val['val_logo_worst'] - best['test_logo_worst'])
print(f'\nvalidation->test gap on the worst fold: {gap:+.1f} pp')
if abs(gap) > 3:
    print('  large gap: the screening winner did not transfer cleanly — prefer a reference config')

d_worst = 100*(best['test_logo_worst'] - off['test_logo_worst'])
d_std   = 100*(best['test_standard_f1'] - off['test_standard_f1'])
print(f'\nvs official five: worst {d_worst:+.1f} pp, standard {d_std:+.1f} pp')
if d_worst > 0.5 and d_std > -0.3:
    print('VERDICT: the searched subset earns promotion — update Vstat and the dimensions downstream')
elif d_worst > 0.5:
    print('VERDICT: robustness gain but a standard-split cost — a judgement call for the supervisor')
else:
    print('VERDICT: no promotion; the official five stand')

ppl_ref = final[final['label']=='REF best + targeted_ppl'].iloc[0]
print(f"\ncost of keeping targeted_ppl on top of the best subset: "
      f"worst {100*(ppl_ref['test_logo_worst']-best['test_logo_worst']):+.1f} pp, "
      f"standard {100*(ppl_ref['test_standard_f1']-best['test_standard_f1']):+.1f} pp")
print('(the proposal names Targeted Perplexity in its title and keywords, so this number decides '
      'whether dropping it needs to be defended or simply reported)')

official five     : std 99.6 | logo mean 98.0 | worst 94.2 (gpt)
best searched     : std 99.6 | logo mean 98.2 | worst 95.2 (gpt)
  features        : burstiness, ttr, quote_ratio, function_word_ratio, passive_ratio, compressibility
  n_active        : 6

validation->test gap on the worst fold: +1.3 pp

vs official five: worst +1.0 pp, standard +0.0 pp
VERDICT: the searched subset earns promotion — update Vstat and the dimensions downstream

cost of keeping targeted_ppl on top of the best subset: worst -1.4 pp, standard +0.2 pp
(the proposal names Targeted Perplexity in its title and keywords, so this number decides whether dropping it needs to be defended or simply reported)


## Notes

- **What is reportable.** Stage-2 test numbers only. Stage-1 validation scores are a search device
  and should never be quoted as results.
- **Promotion changes dimensions everywhere.** If a subset of size k is promoted, Vstat becomes R^k
  and Vhybrid R^(768+k); the feature-order contract, the skills, the work plan, Track B, and the
  thesis figures all need the new numbers.
- **The `targeted_ppl` line is a thesis decision, not just a metric.** The proposal names Targeted
  Perplexity in its title, keywords and objectives. If the measured cost of keeping it is small, keep
  it and preserve the proposal's promise; if the search shows it is expensive, the write-up must
  defend the removal explicitly and the supervisor should approve it.
- **Watch the validation-to-test gap.** A small gap means the two-stage protocol did its job. A large
  one is itself a finding worth a sentence: exhaustive subset search over thousands of candidates
  overfits even a held-out selection split when the folds are small.
- **The marginal table generalizes better than the winner.** For the thesis narrative, "feature X
  contributes +Y pp averaged over thousands of contexts" is a stronger claim than "the best subset
  happened to contain X".